In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "tealeaf").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)


In [2]:
import pandas as pd
import collections
import plotnine as p9
import time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from tealeaf import calcutta
import scipy.sparse as sp
import scipy.io
from importlib import reload
reload(calcutta)

<module 'calcutta' from '/gpfs/commons/home/daknowles/calcutta/calcutta/calcutta.py'>

## Load data

In [3]:
salmon_dir = Path("/gpfs/commons/groups/knowles_lab/data/parse/original_SPLITseq_GSE110823/salmon_spliceu/quant_t2t_dedup")

In [4]:
with open(salmon_dir / "alevin" / "quants_mat_cols.txt", 'r') as file:
    features = np.array([line.strip() for line in file.readlines()])

In [5]:
indexdir = Path("/gpfs/commons/groups/knowles_lab/index/salmon/mus_spliceu/")
transcript_lengths_dic = calcutta.get_transcript_lengths(indexdir / "spliceu.fa")

In [7]:
# Load EC <> transcript map
map_cache = salmon_dir / "alevin" / "ec_transcript_map.npz"

if map_cache.is_file(): 
    ec_transcript_mat = scipy.sparse.load_npz(map_cache)
else:
    num_genes, num_ec, ecs = calcutta.read_alevin_ec(salmon_dir / "alevin" / "gene_eqclass.txt.gz") 
    ecs_list = [ ecs[k] for k in range(len(ecs)) ] # this works because indexing is dense
    ec_transcript_mat = calcutta.to_coo(ecs_list)
    scipy.sparse.save_npz(map_cache, ec_transcript_mat)
    
ec_transcript_mat.shape # ECs x transcripts (418040, 118489) for kallisto, (3987532, 116918) for salmon with spliceu

(3987532, 116918)

In [8]:
# Load cells x EC counts
npz_cache = salmon_dir / "alevin" / "geqc_counts.npz"

if npz_cache.is_file(): 
    cell_ec_sparse = scipy.sparse.load_npz(npz_cache)
else:
    cell_ec_sparse = scipy.io.mmread(salmon_dir / "alevin" / "geqc_counts.mtx") 
    scipy.sparse.save_npz(npz_cache, cell_ec_sparse)    

In [ ]:
# Load cells x transcript counts (don't actually need this I guess?)
counts_cache = salmon_dir / "alevin" / "quants_mat.npz"

if counts_cache.is_file(): 
    count_mat = scipy.sparse.load_npz(counts_cache)
else: 
    count_mat = scipy.io.mmread(salmon_dir / "alevin" / "quants_mat.mtx") # cells x genes
    scipy.sparse.save_npz(counts_cache)    
    
count_mat.shape # 192k (cells) by 117k (transcripts)

In [ ]:
pseudobulk = calcutta.sparse_sum(cell_ec_sparse,0) # ~4M ECs

## EDA

In [ ]:
(pseudobulk==0).sum() # 0! every EC has a read (maybe by definition?)

In [ ]:
(pseudobulk==1).sum() # 2M! 

In [ ]:
ecs_per_transcript = calcutta.sparse_sum(ec_transcript_mat,0)

In [ ]:
(ecs_per_transcript==0).sum() # 10k transcripts are in no EC

In [ ]:
pseudobulk.sum() # 500M 

In [ ]:
(pseudobulk < 5).mean() # 85% of ECs only have less than 5 reads associated with them

In [ ]:
((pseudobulk < 5) & (ec_sizes==1)).mean() # the low count ECs if anything are depleted of unique (size one) sets

In [ ]:
((pseudobulk < 5) & (ec_sizes>1)).mean()

In [ ]:
((pseudobulk >= 5) & (ec_sizes==1)).mean()

In [ ]:
pseudobulk[pseudobulk < 5].sum() # 10% of counts 

In [ ]:
unique_values, counts = np.unique(pseudobulk[pseudobulk <= 30], return_counts=True)

plt.bar(unique_values, counts)

In [ ]:
transcript_count = pseudobulk @ ec_transcript_mat

In [ ]:
(transcript_count==0.).sum() # worth removing these? probably 

In [ ]:
ec_transcript_mat.shape

In [ ]:
plt.hist( np.log10(pseudobulk[pseudobulk >= 10]),30) # very power law-y

In [ ]:
ec_sizes = calcutta.sparse_sum(ec_transcript_mat,1)

In [ ]:
plt.scatter(np.log10(ec_sizes), np.log10(pseudobulk), alpha=0.1)
plt.xlabel("log10(EC size)")
plt.ylabel("log10(UMI count)")

In [ ]:
import plotnine as p9
data = pd.DataFrame({'x': np.log10(ec_sizes), 'y': pseudobulk})

# Discretize x into bins
data['x_bins'] = pd.cut(data['x'], bins=10)  # Adjust the number of bins as per your preference

# Plot the boxplots
p9.ggplot(data, p9.aes(x='x_bins', y='y')) + p9.geom_boxplot(fill='lightblue') + p9.scale_y_log10() 

In [ ]:
from scipy.sparse.csgraph import connected_components
g = ec_transcript_mat.T @ ec_transcript_mat
n_components, labels = connected_components(csgraph=g, directed=False, return_labels=True)
n_components # 19163 for kallisto standard index, 38k here

In [ ]:
pd.Series(labels).value_counts().max() # 101k

In [ ]:
pd.Series(labels).value_counts()

In [ ]:
cell_ec_sparse.shape

## Filter data

In [ ]:
ECs_to_keep = pseudobulk >= 5

ec_transcript_filt = ec_transcript_mat.tocsr()[ECs_to_keep,:]

In [ ]:
cell_ec_filt = cell_ec_sparse.tocsc()[:,ECs_to_keep]

In [ ]:
transcript_count.shape 

In [ ]:
features_to_keep = transcript_count > 0
features_filt = features[features_to_keep]
ec_transcript_ff = ec_transcript_filt[:,features_to_keep]

In [ ]:
pseudobulk_filt = calcutta.sparse_sum(cell_ec_filt,0)
pseudobulk_filt.shape

In [ ]:
from collections import Counter
a = Counter()
for k in transcript_lengths_dic.keys(): 
    split = k.split("-")
    if len(split)==1: 
        a[("mature",k[:7])] += 1
    else: 
        a[(split[1],k[:7])] += 1
a   

In [ ]:
feature_lengths, w = calcutta.get_feature_weights(features_filt, transcript_lengths_dic)

In [ ]:
is_spliced = np.array([ g[:7] == 'ENSMUST' for g in features_filt ])

plt.hist(np.log10(feature_lengths[~is_spliced]), 30, alpha=.3, label = "unspliced")
plt.hist(np.log10(feature_lengths[is_spliced]), 30, alpha=.3, label = "spliced")
plt.legend()
plt.show()

## EM

In [ ]:
alpha = calcutta.EM(pseudobulk_filt, ec_transcript_ff.tocoo(), w)

In [ ]:
neg_hessian_factor, neg_hessian = calcutta.get_neg_hessian(pseudobulk_filt, ec_transcript_ff.tocoo(), w)

In [ ]:
neg_hessian.nnz / np.prod(neg_hessian.shape)

In [ ]:
# TODO: check we are getting sparsity from x. Use hessian.eliminate_zeros()? 

In [ ]:
# x will help make this more sparse than otherwise

In [ ]:
neg_hessian.shape

In [ ]:
neg_hessian.nnz / np.prod(neg_hessian.shape)

In [ ]:
ec_transcript_mat.nnz / np.prod(ec_transcript_mat.shape)

In [ ]:
neg_hessian_csc = neg_hessian.tocsc()

## Uncertainty quantification

In [ ]:
nonz_mean = neg_hessian.data.mean()
"%.4g" % nonz_mean
neg_hessian_scaled = neg_hessian / nonz_mean
#neg_hessian_csc.data /= neg_hessian_csc.data.mean()

In [ ]:
from sksparse.cholmod import cholesky, cholesky_AAt
start_time = time.time()
ch = cholesky(neg_hessian_scaled, beta=1e-8)
time.time() - start_time # 10 seconds(!) 370second=6minutes now!? With filtering: 233s = 3.8m

In [ ]:
nonz_factor_mean = neg_hessian_factor.data.mean()
neg_hessian_factor_scaled = neg_hessian_factor / nonz_factor_mean

In [ ]:
from sksparse.cholmod import cholesky, cholesky_AAt
start_time = time.time()
ch_AAt = cholesky_AAt(neg_hessian_factor_scaled.T, beta=1e-4) # this is slower than chol()
time.time() - start_time

In [ ]:
start_time = time.time()
#inv_hess = sp.linalg.factorized(nhessian_csc + 1e-11 * sp.eye(nhessian_csc.shape[0])) # much slower!
time.time() - start_time # 10minutes

In [ ]:
suppa_se = pd.read_csv("/gpfs/commons/home/daknowles/calcutta/suppa_mouse_basic/suppaout_SE_strict.ioe", sep="\t")
suppa_se

In [ ]:
feat_df = pd.DataFrame({"label":features_filt, "idx":np.arange(len(features_filt))}) # map transcripts to idx
t2t = pd.read_csv(indexdir / "t2t_dedup_v2.tsv",  sep = "\t", names = ["transcript","label"]) # deal with duplicates
merged = t2t.merge(feat_df, on = "label")
tnv2idx = dict(zip(merged["transcript"], merged["idx"])) # convert to dictionary, includes deduped transcripts (but not filtered ones! )

In [ ]:
alt_trans = [ [ tnv2idx.get(trans,-1) for trans in g.split(",") ] for g in suppa_se.alternative_transcripts ]
tot_trans = [ [ tnv2idx.get(trans,-1) for trans in g.split(",") ] for g in suppa_se.total_transcripts ]
has_missing = [ (-1 in g) for g in tot_trans]
prop_w_missing = np.mean( has_missing ) # ok fixed, 0! 
prop_w_missing

In [ ]:
ref_trans = [ list(set(tot) - set(alt)) for alt,tot in zip(alt_trans,tot_trans) ]

In [ ]:
alt_trans_filt = [ [ f for f in g if f!=-1] for g in alt_trans ] # remove filtered transcripts
ref_trans_filt = [ [ f for f in g if f!=-1] for g in ref_trans ]

In [ ]:
# remove events that where the alt and ref set is now empty
alt_len = np.array([ len(g) for g in alt_trans_filt ])
ref_len = np.array([ len(g) for g in ref_trans_filt ])
to_remove = np.logical_and(ref_len==0., alt_len==0.) # and makes more sense here: if alt or ref is empty we will have PSI=0 or PSI=1 but still valid

alt_filt = [ alt_trans_filt[i] for i in np.where(~to_remove)[0] ]
ref_filt = [ ref_trans_filt[i] for i in np.where(~to_remove)[0] ]

In [ ]:

num_transcripts

alt_trans_mat = calcutta.to_coo(alt_filt, shape=(len(alt_filt), len(features_filt))) # event x transcripts
ref_trans_mat = calcutta.to_coo(ref_filt, shape=(len(alt_filt), len(features_filt)))

In [ ]:
alt_tpm = alt_trans_mat @ alpha
ref_tpm = ref_trans_mat @ alpha

In [ ]:
logit_psi = np.log(alt_tpm) - np.log(ref_tpm) # add pseudocount?
logit_psi

In [ ]:
np.isfinite(logit_psi).mean() # mostly non-nan. nan is 0/0 which is missing (e.g. gene not expressed)

In [ ]:
plt.hist(logit_psi[np.isfinite(logit_psi)],30)

In [ ]:
grad_logit_psi = sp.diags(1. / alt_tpm) @ alt_trans_mat - sp.diags(1. / ref_tpm) @ ref_trans_mat

In [ ]:
grad_logit_psi.shape

In [ ]:
# I wonder if this might actually be faster if grad_logit_psi were dense? 
start_time = time.time()
Linv_grad = ch.solve_L(grad_logit_psi.T , use_LDLt_decomposition = False)
time.time() - start_time # ~6minutes for 17k events. Now 2356s = 40min. With filtering -> 28min. 

In [ ]:
Linv_grad2 = Linv_grad.copy()
Linv_grad2.data *= Linv_grad2.data

var_logitPSI = calcutta.sparse_sum(Linv_grad2,0) * nonz_mean # variance for event logitPSI 
# TODO: fix scaling 

In [ ]:
# I guess the difference is the kallisto version didn't include precursor? 

In [ ]:
eps = 1e-8
P = neg_hessian_scaled.shape[0]
sp_eps = sp.diags(np.full(P,eps))
start_time = time.time()
invA_b_spsolve = scipy.sparse.linalg.spsolve(neg_hessian_scaled + sp_eps, grad_logit_psi.T) 
var_logitPSI_2 = grad_logit_psi.multiply(invA_b_spsolve).sum(0) * nonz_mean
time.time() - start_time

In [ ]:
var_logitPSI_2

In [ ]:
1

In [ ]:
1